<a href="https://colab.research.google.com/github/OscarGarcia1995/OscarGarcia1995/blob/main/Data_book_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

In [8]:
df = pd.read_csv('/content/best_sellin_books_total.csv', encoding='latin1')

In [9]:
# ── Load & Clean ──────────────────────────────────────────────────────────────
df = pd.read_csv('best_sellin_books_total.csv', encoding='latin1')
df['rating_num'] = df['Rating'].str.extract(r'([\d.]+)').astype(float)
df['price_num']  = df['price'].str.replace('$','').str.strip().astype(float)

def parse_rank(s):
    try: return int(str(s).replace('#','').strip())
    except: return np.nan

df['rank_2023'] = df['id_2023'].apply(parse_rank)
df['rank_2024'] = df['id_2024'].apply(parse_rank)
df['rank_2025'] = df['id_2025'].apply(parse_rank)
df['in_years'] = df[['rank_2023','rank_2024','rank_2025']].notna().sum(axis=1)

top_authors = df.groupby('Author').size().sort_values(ascending=False).head(10).reset_index()
top_authors.columns = ['Author','count']

genre_stats = df.groupby('Genre').agg(
    books=('Book name','count'),
    avg_rating=('rating_num','mean'),
    avg_reviews=('reviews count','mean'),
    avg_price=('price_num','mean')
).sort_values('books', ascending=False).head(10)

In [10]:
# ── Style ─────────────────────────────────────────────────────────────────────
BG      = '#0F1117'
CARD    = '#1A1D27'
ACCENT1 = '#6C63FF'
ACCENT2 = '#FF6584'
ACCENT3 = '#43C6AC'
ACCENT4 = '#FFD166'
TEXT    = '#E8E8F0'
SUBTEXT = '#9090A8'
GRID    = '#2A2D3E'

plt.rcParams.update({
    'figure.facecolor': BG,  'axes.facecolor': CARD,
    'axes.edgecolor':   GRID,'axes.labelcolor': TEXT,
    'xtick.color':  SUBTEXT, 'ytick.color':    SUBTEXT,
    'text.color':   TEXT,    'grid.color':     GRID,
    'grid.linestyle': '--',  'grid.alpha':     0.5,
})

fig = plt.figure(figsize=(24, 26), facecolor=BG)
fig.text(0.5, 0.976, 'Amazon Best-Selling Books  ·  Data Analyst Dashboard',
         ha='center', va='top', fontsize=26, fontweight='bold', color=TEXT)
fig.text(0.5, 0.963, '210 titles  ·  2023–2025  ·  13 variables',
         ha='center', va='top', fontsize=13, color=SUBTEXT)

gs = fig.add_gridspec(4, 3, hspace=0.48, wspace=0.35,
                      top=0.952, bottom=0.04, left=0.06, right=0.97)

In [11]:
# ── 0. KPI Cards ──────────────────────────────────────────────────────────────
ax_kpi = fig.add_subplot(gs[0, 0])
ax_kpi.set_xlim(0, 1); ax_kpi.set_ylim(0, 1)
ax_kpi.axis('off'); ax_kpi.set_facecolor(BG)

kpis = [
    ("210",                              "Total Books",    ACCENT1),
    (f"{df['rating_num'].mean():.2f} ★", "Avg Rating",    ACCENT3),
    (f"${df['price_num'].mean():.2f}",   "Avg Price",     ACCENT4),
    (f"{int(df['reviews count'].median()):,}", "Median Reviews", ACCENT2),
]
for i, (val, lab, col) in enumerate(kpis):
    x = 0.13 + (i % 2) * 0.52
    y = 0.72 - (i // 2) * 0.44
    rect = mpatches.FancyBboxPatch(
        (x - 0.09, y - 0.15), 0.38, 0.32,
        boxstyle="round,pad=0.02", facecolor=CARD, edgecolor=col, linewidth=1.5)
    ax_kpi.add_patch(rect)
    ax_kpi.text(x + 0.10, y + 0.07, val, ha='center', fontsize=16,
                fontweight='bold', color=col)
    ax_kpi.text(x + 0.10, y - 0.06, lab, ha='center', fontsize=9, color=SUBTEXT)
ax_kpi.set_title('Key Metrics', color=TEXT, fontsize=13, fontweight='bold', pad=10)

Text(0.5, 1.0, 'Key Metrics')

In [12]:
# ── 1. Genre Donut ────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 1])
colors_pie = [ACCENT1, ACCENT2, ACCENT3, ACCENT4, '#F77F00',
              '#7BF1A8', '#C9CBFF', '#FF9A8B', '#A8EDEA', '#FED6E3']
wedges, _, autotexts = ax1.pie(
    genre_stats['books'], labels=None, autopct='%1.0f%%',
    colors=colors_pie, startangle=140,
    wedgeprops=dict(width=0.55, edgecolor=BG, linewidth=2), pctdistance=0.75)
for at in autotexts:
    at.set_fontsize(8); at.set_color(BG); at.set_fontweight('bold')
ax1.legend(genre_stats.index, loc='lower center', bbox_to_anchor=(0.5, -0.22),
           fontsize=7, ncol=2, frameon=False, labelcolor=SUBTEXT)
ax1.set_title('Genre Distribution (Top 10)', color=TEXT, fontsize=13, fontweight='bold')

Text(0.5, 1.0, 'Genre Distribution (Top 10)')

In [13]:
# ── 2. Avg Reviews by Genre ───────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
gs2 = genre_stats['avg_reviews'].sort_values()
bars = ax2.barh(gs2.index, gs2.values, color=ACCENT1, alpha=0.85, height=0.6)
for bar in bars:
    ax2.text(bar.get_width() + 2000, bar.get_y() + bar.get_height() / 2,
             f'{bar.get_width():,.0f}', va='center', fontsize=8, color=SUBTEXT)
ax2.set_xlabel('Avg Reviews', color=SUBTEXT, fontsize=10)
ax2.set_title('Avg Reviews by Genre', color=TEXT, fontsize=13, fontweight='bold')
ax2.grid(axis='x'); ax2.tick_params(labelsize=8)
ax2.set_xlim(0, gs2.max() * 1.2)

(0.0, 270017.15294117643)

In [14]:
# ── 3. Rating Distribution ────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(df['rating_num'], bins=14, color=ACCENT3, edgecolor=BG, alpha=0.9)
ax3.axvline(df['rating_num'].mean(), color=ACCENT2, linestyle='--', linewidth=2,
            label=f"Mean {df['rating_num'].mean():.2f}")
ax3.axvline(df['rating_num'].median(), color=ACCENT4, linestyle=':', linewidth=2,
            label=f"Median {df['rating_num'].median():.2f}")
ax3.legend(fontsize=9, frameon=False)
ax3.set_xlabel('Rating', color=SUBTEXT)
ax3.set_ylabel('Count',  color=SUBTEXT)
ax3.set_title('Rating Distribution', color=TEXT, fontsize=13, fontweight='bold')

Text(0.5, 1.0, 'Rating Distribution')

In [15]:
# ── 4. Scatter: Rating vs log(Reviews) colored by Price ───────────────────────
ax4 = fig.add_subplot(gs[1, 1])
valid = df.dropna(subset=['rating_num', 'reviews count', 'price_num'])
sc = ax4.scatter(valid['rating_num'], np.log1p(valid['reviews count']),
                 c=valid['price_num'], cmap='plasma', alpha=0.7, s=40, edgecolors='none')
cb = plt.colorbar(sc, ax=ax4)
cb.set_label('Price ($)', color=SUBTEXT, fontsize=9)
cb.ax.yaxis.set_tick_params(color=SUBTEXT)
plt.setp(cb.ax.yaxis.get_ticklabels(), color=SUBTEXT, fontsize=8)

X  = valid[['rating_num']]
y  = np.log1p(valid['reviews count'])
lr = LinearRegression().fit(X, y)
xr = np.linspace(X['rating_num'].min(), X['rating_num'].max(), 100)
ax4.plot(xr, lr.predict(xr.reshape(-1, 1)), color=ACCENT2, linewidth=2, label='Trend')
ax4.legend(fontsize=9, frameon=False)
ax4.set_xlabel('Rating',       color=SUBTEXT)
ax4.set_ylabel('log(Reviews)', color=SUBTEXT)
ax4.set_title('Rating vs Reviews (color = Price)', color=TEXT, fontsize=13, fontweight='bold')

Text(0.5, 1.0, 'Rating vs Reviews (color = Price)')

In [16]:
# ── 5. Price Boxplot by Format ────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
main_forms = ['Paperback', 'Hardcover', 'Board book']
form_data  = [df[df['form'] == f]['price_num'].dropna() for f in main_forms]
bp = ax5.boxplot(form_data, patch_artist=True, widths=0.45,
                 medianprops=dict(color=BG, linewidth=2))
for patch, col in zip(bp['boxes'], [ACCENT1, ACCENT2, ACCENT3]):
    patch.set_facecolor(col); patch.set_alpha(0.8)
for w in bp['whiskers']: w.set(color=SUBTEXT, linewidth=1.2)
for c in bp['caps']:     c.set(color=SUBTEXT, linewidth=1.2)
for f in bp['fliers']:   f.set(marker='o', color=ACCENT4, alpha=0.5, markersize=4)
ax5.set_xticklabels(main_forms, fontsize=10)
ax5.set_ylabel('Price ($)', color=SUBTEXT)
ax5.set_title('Price Distribution by Format', color=TEXT, fontsize=13, fontweight='bold')
ax5.grid(axis='y')


In [17]:
# ── 6. Top 10 Authors ─────────────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, :2])
pal = [ACCENT1 if i < 3 else ACCENT3 for i in range(len(top_authors))]
bars6 = ax6.bar(top_authors['Author'], top_authors['count'],
                color=pal, edgecolor=BG, width=0.6)
for bar in bars6:
    ax6.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
             str(int(bar.get_height())), ha='center', fontsize=10,
             color=ACCENT4, fontweight='bold')
ax6.set_ylabel('# Best-Seller Titles', color=SUBTEXT)
ax6.set_title('Top 10 Authors by Best-Seller Titles (purple = top 3)',
              color=TEXT, fontsize=13, fontweight='bold')
ax6.tick_params(axis='x', rotation=30, labelsize=9)
ax6.grid(axis='y')
ax6.set_ylim(0, top_authors['count'].max() + 1.5)

(0.0, 11.5)

In [18]:
# ── 7. Year Presence Donut ────────────────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 2])
yoy = df['in_years'].value_counts().sort_index()
labels_yoy = {1: '1 year only', 2: '2 years', 3: 'All 3 years'}
cols_yoy   = [ACCENT2, ACCENT4, ACCENT3]
_, _, auto7 = ax7.pie(
    yoy.values, labels=[labels_yoy[k] for k in yoy.index],
    autopct='%1.1f%%', colors=cols_yoy,
    wedgeprops=dict(width=0.55, edgecolor=BG, linewidth=2),
    pctdistance=0.75, textprops=dict(color=TEXT, fontsize=10))
for at in auto7:
    at.set_fontsize(9); at.set_color(BG); at.set_fontweight('bold')
ax7.set_title('Books in 1 / 2 / 3 Years on List', color=TEXT, fontsize=13, fontweight='bold')

Text(0.5, 1.0, 'Books in 1 / 2 / 3 Years on List')

In [19]:
# ── 8. Heatmap: Avg Rating × Genre × Format ───────────────────────────────────
ax8 = fig.add_subplot(gs[3, :2])
pivot = df[df['form'].isin(main_forms)].pivot_table(
    values='rating_num', index='Genre', columns='form', aggfunc='mean')
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index[:10]]
sns.heatmap(pivot, ax=ax8, cmap='YlOrRd', annot=True, fmt='.2f',
            linewidths=0.5, linecolor=BG, cbar_kws={'shrink': 0.7},
            annot_kws={'size': 9}, vmin=4.0, vmax=5.0)
ax8.set_title('Avg Rating by Genre × Format', color=TEXT, fontsize=13, fontweight='bold')
ax8.set_xlabel('Format', color=SUBTEXT)
ax8.set_ylabel('', color=SUBTEXT)
ax8.tick_params(colors=SUBTEXT, labelsize=8)
plt.setp(fig.axes[-1].yaxis.get_ticklabels(), color=SUBTEXT)

[None, None, None, None, None, None]

In [20]:
# ── 9. Review Counts by Years on List ────────────────────────────────────────
ax9 = fig.add_subplot(gs[3, 2])
for yr, col in zip([1, 2, 3], [ACCENT2, ACCENT4, ACCENT3]):
    subset = df[df['in_years'] == yr]['reviews count']
    ax9.hist(subset, bins=20, alpha=0.6, color=col,
             label=f'{yr} yr(s)  n={len(subset)}', edgecolor='none')
ax9.set_xlabel('Review Count', color=SUBTEXT)
ax9.set_ylabel('Frequency',    color=SUBTEXT)
ax9.set_title('Review Counts by Years on List', color=TEXT, fontsize=13, fontweight='bold')
ax9.legend(fontsize=9, frameon=False)
ax9.grid(axis='y')

In [21]:
plt.savefig('books_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
print("✓ Dashboard saved as books_dashboard.png")

✓ Dashboard saved as books_dashboard.png
